# Scalable Parallel K-Means (MPI + CUDA)
### Course: Parallel Computer Architecture & Programming (PS-5)
**Team:** Tirth Patel (Distributed MPI Architecture) & Aarush / Dispo (CUDA Kernel Acceleration)

This Google Colab notebook runs the complete scalable parallel K-Means clustering pipeline with hardware GPU acceleration on NVIDIA GPUs.

> **Important:** Ensure your Colab runtime has a GPU: Go to `Runtime` -> `Change runtime type` -> select **T4 GPU**.

In [ ]:
# Step 1: Verify NVIDIA GPU & CUDA environment
!nvidia-smi
!nvcc --version

## Step 2: Install System Dependencies
Google Colab instances require OpenMPI runtime and development headers (`openmpi-bin`, `libopenmpi-dev`).

In [ ]:
!apt-get update -qq
!apt-get install -y openmpi-bin libopenmpi-dev
!pip install --quiet numpy scipy matplotlib pandas tabulate

## Step 3: Clone Repository (Branch `colab`)
Clone the repository or navigate to the project directory.

In [ ]:
import os
if not os.path.exists('src/kmeans_mpi.cpp'):
    !git clone -b colab https://github.com/aarush-mpl/PCAP_project.git
    %cd PCAP_project
else:
    print('Already inside project directory:', os.getcwd())

!ls -la

## Step 4: Compile All Targets
Compiles:
1. `bin/kmeans_serial` (CPU reference baseline)
2. `bin/kmeans_mpi` (Distributed CPU assignment)
3. `bin/kmeans_hybrid` (MPI distributed master/worker + **NVIDIA CUDA GPU assignment kernel**)

In [ ]:
!make clean
!make all
!make hybrid
!ls -lh bin/

## Step 5: Generate Benchmark Dataset
Generates synthetic dataset with 100,000 points, 16 dimensions, and 10 natural clusters.

In [ ]:
!mkdir -p data results
!python3 scripts/generate_dataset.py --n 100000 --d 16 --k 10 --out data/synth_100k.bin --seed 42
!ls -lh data/

## Step 6: Benchmark Execution
Run all 3 versions using identical seed (42) and initialization method (`kmeans++`) for strict comparison.

In [ ]:
# 6a: Serial CPU Reference Baseline
!./bin/kmeans_serial data/synth_100k.bin 10 100 1e-4 42 results/colab_serial kmeans++

In [ ]:
# 6b: Distributed MPI CPU (2 ranks)
!mpirun --allow-run-as-root --oversubscribe -np 2 ./bin/kmeans_mpi data/synth_100k.bin 10 100 1e-4 42 results/colab_mpi kmeans++

In [ ]:
# 6c: Hybrid MPI + CUDA GPU Accelerated (2 ranks)
# NOTE: In Google Colab with GPU runtime, this runs directly on the NVIDIA Tesla GPU!
!mpirun --allow-run-as-root --oversubscribe -np 2 ./bin/kmeans_hybrid data/synth_100k.bin 10 100 1e-4 42 results/colab_hybrid kmeans++

## Step 7: Mathematical Validation
Strictly verify that MPI and Hybrid implementations produce matching centroids and SSE to the Serial baseline.

In [ ]:
print('=== 1. Validating MPI vs Serial ===')
!python3 scripts/validate_correctness.py results/colab_serial results/colab_mpi
print('\n=== 2. Validating CUDA Hybrid vs Serial ===')
!python3 scripts/validate_correctness.py results/colab_serial results/colab_hybrid

## Step 8: Performance & Speedup Comparison
Extract metrics and display runtime breakdown and speedup charts.

In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd

def parse_metrics(filepath):
    data = {}
    if not os.path.exists(filepath):
        print(f'Warning: File not found: {filepath}')
        return data
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(None, 1)
            if len(parts) == 2:
                k, v = parts[0], parts[1]
                try:
                    data[k] = float(v)
                except ValueError:
                    data[k] = v
    return data

serial = parse_metrics('results/colab_serial_metrics.txt')
mpi = parse_metrics('results/colab_mpi_metrics.txt')
hybrid = parse_metrics('results/colab_hybrid_metrics.txt')

s_time = serial.get('total_time_s', 0.0)
m_time = mpi.get('total_time_s', 0.0)
h_time = hybrid.get('total_time_s', 0.0)

s_comp = serial.get('compute_time_s', 0.0)
m_comp = mpi.get('compute_time_avg_s', mpi.get('compute_time_s', 0.0))
h_comp = hybrid.get('compute_time_avg_s', hybrid.get('compute_time_s', 0.0))

N_val = int(serial.get('N', mpi.get('N', hybrid.get('N', 0))))
D_val = int(serial.get('D', mpi.get('D', hybrid.get('D', 0))))
K_val = int(serial.get('K', mpi.get('K', hybrid.get('K', 0))))

m_speedup = (s_time / m_time) if m_time > 0 else 1.0
h_speedup = (s_time / h_time) if h_time > 0 else 1.0

df = pd.DataFrame([
    {'Mode': 'Serial (CPU 1 core)', 'Total Time (s)': s_time, 'Compute Time (s)': s_comp, 'Speedup': 1.0},
    {'Mode': 'MPI (CPU 2 ranks)', 'Total Time (s)': m_time, 'Compute Time (s)': m_comp, 'Speedup': m_speedup},
    {'Mode': 'CUDA Hybrid (GPU + MPI)', 'Total Time (s)': h_time, 'Compute Time (s)': h_comp, 'Speedup': h_speedup}
])

print(f'=== Performance Summary (N={N_val:,}, D={D_val}, K={K_val}) ===')
print(df.to_string(index=False))

# Plot comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
modes = ['Serial', 'MPI', 'CUDA Hybrid']
times = [s_time, m_time, h_time]
speedups = [1.0, m_speedup, h_speedup]
colors = ['#4C72B0', '#55A868', '#C44E52']

max_t = max(times) if max(times) > 0 else 1.0
ax1.bar(modes, times, color=colors)
ax1.set_ylabel('Total Time (seconds)')
ax1.set_title(f'Execution Time — N={N_val:,}, D={D_val} (Lower is Better)')
ax1.set_ylim(0, max_t * 1.25)
for i, v in enumerate(times):
    ax1.text(i, v + 0.02 * max_t, f'{v:.4f}s', ha='center', fontweight='bold')

max_s = max(speedups) if max(speedups) > 0 else 1.0
ax2.bar(modes, speedups, color=colors)
ax2.set_ylabel('Speedup vs Serial')
ax2.set_title(f'Speedup Factor — N={N_val:,}, D={D_val} (Higher is Better)')
ax2.set_ylim(0, max_s * 1.25)
for i, v in enumerate(speedups):
    ax2.text(i, v + 0.02 * max_s, f'{v:.2f}x', ha='center', fontweight='bold')

plt.tight_layout()
os.makedirs('results', exist_ok=True)
plt.savefig('results/colab_speedup_plot.png', dpi=150)
plt.show()


## Step 9: (Optional) Automated End-to-End Pipeline
You can also run the full end-to-end automated pipeline script with a single command:
```bash
!bash run_colab.sh
```